# IDSIA Melon results

Load a family (or all) from `out/runs/`. If `out/eval/` is missing, run the test set and save MAE at h=1 / 10 / 50.

In [ ]:
from pathlib import Path
import json

from quadrotor.geometry import train_device
from quadrotor.idsia.baselines import load_phys_res, load_physics
from quadrotor.idsia.data import load_idsia_prepared
from quadrotor.idsia.evaluate import eval_model, family_summary, pack_mae
from quadrotor.idsia.recipe import EVAL, FAMILIES, KEYS
from quadrotor.idsia.train_lib import list_seeds, load_trained
from quadrotor.evaluate import benchmark_mae
from quadrotor.idsia.recipe import HORIZON
from quadrotor.idsia.evaluate import save_eval

FAMILY = "all"  # or "quad_dynamics" / "phys_quotient" / "phys_quotient_sym"
DEVICE = train_device("cpu")
_train, _val, test = load_idsia_prepared()
families = FAMILIES if FAMILY == "all" else (FAMILY,)

In [ ]:
def cell(values):
    return " / ".join(f"{v:.4f}" for v in values)

phys = load_physics(DEVICE)
mae_p = pack_mae(benchmark_mae(phys, test, horizon=HORIZON, start_stride=1))
save_eval("physics", 0, benchmark_mae(phys, test, horizon=HORIZON, start_stride=1))
pub = load_phys_res(device=DEVICE)
mae_r = pack_mae(benchmark_mae(pub, test, horizon=HORIZON, start_stride=1))
save_eval("phys_res", 0, benchmark_mae(pub, test, horizon=HORIZON, start_stride=1))
print(f"{'model':22s}  {'p':28s}  {'R':28s}  {'v':28s}  {'ω'}")
print(f"{'physics':22s}  {cell(mae_p['position']):28s}  {cell(mae_p['rotation']):28s}  {cell(mae_p['velocity']):28s}  {cell(mae_p['omega'])}")
print(f"{'phys+res':22s}  {cell(mae_r['position']):28s}  {cell(mae_r['rotation']):28s}  {cell(mae_r['velocity']):28s}  {cell(mae_r['omega'])}")

for family in families:
    seeds = list_seeds(family)
    if not seeds:
        print(f"{family:22s}  no checkpoints")
        continue
    for seed in seeds:
        mae_path = EVAL / family / f"seed_{seed}" / "mae.json"
        if mae_path.is_file():
            row = json.loads(mae_path.read_text())
            packed = {k: row[k] for k in KEYS}
        else:
            model = load_trained(family, seed, device=DEVICE)
            packed = eval_model(model, test, family, seed)
        print(f"{family+' s'+str(seed):22s}  {cell(packed['position']):28s}  {cell(packed['rotation']):28s}  {cell(packed['velocity']):28s}  {cell(packed['omega'])}")
    if len(seeds) > 1:
        summary = family_summary(family)
        mean = {k: summary[k]["mean"] for k in KEYS}
        print(f"{family+' mean':22s}  {cell(mean['position']):28s}  {cell(mean['rotation']):28s}  {cell(mean['velocity']):28s}  {cell(mean['omega'])}")